In [4]:
import re
import time
from pathlib import Path
from typing import List, Dict
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import asyncio
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy

In [5]:
def extract_main_html_content(html: str) -> str:
	soup = BeautifulSoup(html, "html.parser")

	# Try main content tags first
	main = soup.find("main")
	if main:
		return main.get_text(separator="\n", strip=True)

	article = soup.find("article")
	if article:
		return article.get_text(separator="\n", strip=True)

	# Fallback: find largest <div> not known to be junk
	candidates = []
	for div in soup.find_all("div"):
		class_names = " ".join(div.get("class", []))
		if any(x in class_names.lower() for x in ["nav", "footer", "header", "cookie", "modal", "popup"]):
			continue
		text_len = len(div.get_text(strip=True))
		if text_len > 200:  # heuristic: only large divs
			candidates.append((text_len, div))

	if candidates:
		# Return text from the largest good div
		return max(candidates, key=lambda x: x[0])[1].get_text(separator="\n", strip=True)

	# Last resort: return full page text (minus scripts/styles)
	for tag in soup(["script", "style", "noscript"]):
		tag.decompose()
	return soup.get_text(separator="\n", strip=True)

In [6]:
import asyncio
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy

class CrawlThemeExtractor:
	CATEGORY_KEYWORDS = {
		"ABOUT_US": ["about", "who-we-are", "company", "our-story", "mission", "vision"],
		"EBOOK": ["ebook", "e-book", "downloads", "whitepaper", "guide", "brochure"],
		"COURSES": ["course", "training", "academy", "learning", "bootcamp"],
		"RECENT_BLOG": ["blog", "insights", "articles", "news", "stories"],
		"TESTIMONIALS": ["testimonial", "reviews", "feedback", "case-studies", "customers"],
		"WEBINAR": ["webinar", "events", "live", "sessions", "recording"],
		"SERVICES": ["service", "solutions", "offerings", "capabilities"],
		"PODCAST": ["podcast", "episodes", "listen", "audio"],
		"SHOP": ["shop", "store", "buy", "product", "checkout", "cart"]
	}

	def __init__(self, max_depth=1):
		self.max_depth = max_depth

	def _get_keywords(self, category: str):
		return self.CATEGORY_KEYWORDS.get(category.upper(), [])

	def _extract_summary(self, markdown: str, lines=3):
		all_lines = markdown.splitlines()
		non_empty = [l.strip() for l in all_lines if l.strip()]
		return "\n".join(non_empty[:lines])

	async def extract_thematic_pages(self, main_url: str, category: str):
		keywords = self._get_keywords(category)
		if not keywords:
			print(f"❌ No keywords defined for category '{category}'")
			return

		config = CrawlerRunConfig(
			deep_crawl_strategy=BFSDeepCrawlStrategy(
				max_depth=self.max_depth,
				include_external=False
			),
			verbose=False
		)

		print(f"🔍 Crawling {main_url} for category: {category} ({keywords})")
		async with AsyncWebCrawler() as crawler:
			results = await crawler.arun(main_url, config=config)

		filtered = [
			r for r in results if any(k in r.url.lower() for k in keywords)
		]

		if not filtered:
			print("⚠️ No relevant subpages found.")
			return

		print(f"✅ Found {len(filtered)} matching subpage(s):\n")
		for r in filtered:
			print(f"🔗 {r.url}")
			if r.html:
				clean_text = extract_main_html_content(r.html)
				print(f"📝 {clean_text[:1000]}")
			else:
				print("⚠️ No HTML content available.")
			print("-" * 50)


In [9]:
extractor = CrawlThemeExtractor()
result = await extractor.extract_thematic_pages("https://redkiteproject.com", "SERVICES")

print(result)

🔍 Crawling https://redkiteproject.com for category: SERVICES (['service', 'solutions', 'offerings', 'capabilities'])


[INIT].... → Crawl4AI 0.6.3 

✅ Found 1 matching subpage(s):

🔗 https://www.redkiteproject.com/services
📝 OUR SERVICES
Services
We offer several ways to access long lasting impactful shifts to performance, safety, and customer service.
Systems Design/Policy Shift
By examining organizational policy and workplace systems, we identify gaps that cost your organization productivity and the bottom line.
Systems analysis, listening projects, and  complexity science are used to support long lasting shifts. Positive outcomes shouldn't end when the consultant leaves.
Post-War Training
Red Kite Project has been utilizing methods built for war zones to help workforces prepare for chaotic conditions since 2008.
​
Behavioral Science and workforce training come together with our blend of 'real world' training to prepare your employees to stay safe and provide top performance even under extreme pressure.
E-Learning
Not every organization can pull their folks in for training. Time or financial constraints don't have to prevent your

In [ ]:
FIRECRAWL_API="fc-29599096ac8b426dbf178180c53500ed"
COLUMN_TO_READ_URL_FROM = "G"
CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1QtKOB5ChRemg2_wJOxeZhn1qao1ZrRjVK8nExVzbxEI/edit?gid=2011509251#gid=2011509251" 
COLUMN_TO_WRITE_URL_TO = {
	"ABOUT_US": "M",
	"EBOOK": "N",
	"COURSES": "O",
	"RECENT_BLOG": "P",
	"TESTIMONIALS": "Q",
	"WEBINAR": "R",
	"SERVICES": "S",
	"PODCAST": "T",
	"SHOP": "U"
}
COLUMN_TO_PROCESS = "RECENT_BLOG" 
CATEGORY_KEYWORDS = {
	"ABOUT_US": ["about", "who-we-are", "company", "our-story", "mission", "vision"],
	"EBOOK": ["ebook", "e-book", "downloads", "whitepaper", "guide", "brochure"],
	"COURSES": ["course", "training", "academy", "learning", "bootcamp"],
	"RECENT_BLOG": ["blog", "insights", "articles", "news", "stories"],
	"TESTIMONIALS": ["testimonial", "reviews", "feedback", "case-studies", "customers"],
	"WEBINAR": ["webinar", "events", "live", "sessions", "recording"],
	"SERVICES": ["service", "solutions", "offerings", "capabilities"],
	"PODCAST": ["podcast", "episodes", "listen", "audio"],
	"SHOP": ["shop", "store", "buy", "product", "checkout", "cart"]
}
def get_url_filter_for_category(category):
	keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
	return lambda url: any(k in url.lower() for k in keywords)

In [ ]:
class Config:
	"""Configuration settings for scraping."""
	max_retries: int = 3
	request_timeout: int = 30
	delay_between_requests: float = 2.0
	max_content_length: int = 10000

# Initialize config
config = Config()

In [ ]:
class GoogleSheetsManager:
	"""Manages interactions with Google Sheets."""
	
	def __init__(self, credentials_file: str):
		self.credentials_file = credentials_file
		self.service = self._setup_service()
	
	def _setup_service(self):
		"""Initialize Google Sheets API service."""
		if not Path(self.credentials_file).exists():
			raise FileNotFoundError(f"Credentials file not found: {self.credentials_file}")
		
		scopes = ['https://www.googleapis.com/auth/spreadsheets']
		creds = service_account.Credentials.from_service_account_file(
			self.credentials_file, scopes=scopes
		)
		return build('sheets', 'v4', credentials=creds)
	
	def extract_spreadsheet_id(self, sheet_url: str) -> str:
		"""Extract spreadsheet ID from Google Sheets URL."""
		pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
		match = re.search(pattern, sheet_url)
		if match:
			return match.group(1)
		raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")
	
	def get_urls(self, spreadsheet_id: str) -> List[str]:
		"""Retrieve URLs from Google Sheet dynamically."""
		range_name = f"{COLUMN_TO_READ_URL_FROM}:{COLUMN_TO_READ_URL_FROM}"
		try:
			result = self.service.spreadsheets().values().get(
					spreadsheetId=spreadsheet_id,
					range=range_name
			).execute()
			values = result.get('values', [])
			start_row = int(range_name.split(':')[0][1:]) if range_name[1].isdigit() else 2
			urls_with_rows = [(start_row + i, row[0]) for i, row in enumerate(values) if row and row[0].strip()]
			print(f"Found {len(urls_with_rows)} URLs to scrape")
			return urls_with_rows
		except Exception as e:
				print(f"Error fetching URLs: {e}")
				return []
	
	def update_results(self, spreadsheet_id: str, results: List[Dict]):
		"""Update Google Sheet with scraping results, using dynamic column based on COLUMN_TO_PROCESS."""
		if not results:
			return

		column_letter = COLUMN_TO_WRITE_URL_TO.get(COLUMN_TO_PROCESS)
		if not column_letter:
			print(f"Invalid COLUMN_TO_PROCESS: {COLUMN_TO_PROCESS}")
			return

		max_retries = config.max_retries
		for attempt in range(max_retries):
			try:
				content_data = [[result['content']] for result in results]
				# You can include status column logic here if needed

				self.service.spreadsheets().values().update(
					spreadsheetId=spreadsheet_id,
					range=f"{column_letter}2:{column_letter}{len(content_data) + 1}",
					valueInputOption='RAW',
					body={'values': content_data}
				).execute()

				print(f"Updated {len(results)} rows in column {column_letter}")
				return

			except HttpError as e:
				print(f"HTTP Error on attempt {attempt + 1}: {e}")
				if e.resp.status == 429:
					print("Rate limit exceeded.")
			except Exception as e:
				print(f"Error on attempt {attempt + 1}: {e}")

			if attempt < max_retries - 1:
				wait_time = 5 * (attempt + 1)
				print(f"Retrying in {wait_time} seconds...")
				time.sleep(wait_time)
			else:
				print("All retries failed. Could not update sheet.")

In [ ]:
reader = GoogleSheetsManager(CREDENTIALS_FILE)
spreadsheet_id = reader.extract_spreadsheet_id(GOOGLE_SHEET_URL)
main_urls_with_rows = reader.get_urls(spreadsheet_id)

In [ ]:
class Crawl4AIExtractor:
	def __init__(self, urls: List[str], max_concurrent: int = 5):
		self.urls = urls
		self.sem = asyncio.Semaphore(max_concurrent)

	async def extract_all_urls(self) -> List[Dict]:
		"""Extract content from all URLs concurrently with rate limiting."""
		if not self.urls:
			print("⚠️ No URLs provided for extraction")
			return [{"status": "fail", "reason": "No URLs provided."}]
		
		async def crawl_url(url):
			async with self.sem:
				try:
					async with AsyncWebCrawler() as crawler:
						result = await crawler.arun(url=url)
						return {
							"url": url,
							"status": "success",
							"content": result.html,
							"title": getattr(result, "title", None),
							"url_type": getattr(result, "url_type", None)
						}
				except Exception as e:
					return {
						"url": url,
						"status": "fail",
						"error": str(e)
					}

		tasks = [crawl_url(url) for url in self.urls]
		results = await asyncio.gather(*tasks)
		success_count = sum(1 for r in results if r["status"] == "success")
		return results
	
	async def extract_first_url(self) -> Dict:
		if not self.urls:
			return {"status": "fail", "reason": "No URLs provided."}
		
		url = self.urls[0]
		try:
			async with AsyncWebCrawler() as crawler:
				result = await crawler.arun(url=url)
				return {
					"url": url,
					"status": "success",
					"content": result.html,
					"title": getattr(result, "title", None),
					"url_type": getattr(result, "url_type", None)
				}
		except Exception as e:
			return {
				"url": url,
				"status": "fail",
				"error": str(e)
			}
		
	async def crawl_main_with_theme_filter(main_url: str):
		config = CrawlerRunConfig(
				deep_crawl_strategy=BFSDeepCrawlStrategy(
						max_depth=1,
						include_external=False,
						url_filter=theme_url_filter
				),
				verbose=True
		)

		async with AsyncWebCrawler() as crawler:
				results = await crawler.arun(main_url, config=config)
				print(f"\n🔍 Found {len(results)} themed subpages from {main_url}\n")

				for result in results:
						print(f"📄 URL: {result.url}")
						print(f"🧾 Content Preview: {result.markdown[:200]}\n")

In [ ]:
def extract_main_html_content(html: str) -> str:
	soup = BeautifulSoup(html, "html.parser")

	# Try main content tags first
	main = soup.find("main")
	if main:
		return main.get_text(separator="\n", strip=True)

	article = soup.find("article")
	if article:
		return article.get_text(separator="\n", strip=True)

	# Fallback: find largest <div> not known to be junk
	candidates = []
	for div in soup.find_all("div"):
		class_names = " ".join(div.get("class", []))
		if any(x in class_names.lower() for x in ["nav", "footer", "header", "cookie", "modal", "popup"]):
			continue
		text_len = len(div.get_text(strip=True))
		if text_len > 200:  # heuristic: only large divs
			candidates.append((text_len, div))

	if candidates:
		# Return text from the largest good div
		return max(candidates, key=lambda x: x[0])[1].get_text(separator="\n", strip=True)

	# Last resort: return full page text (minus scripts/styles)
	for tag in soup(["script", "style", "noscript"]):
		tag.decompose()
	return soup.get_text(separator="\n", strip=True)

In [ ]:
async def process_main_url(row_index: int, main_url: str, category: str, api_key: str = None):
	"""Use Crawl4AI to go 1 level deep and extract relevant subpage content for a category."""
	try:
		print(f"🔍 Crawling {main_url} for category {category}...")

		url_filter = get_url_filter_for_category(category)
		config = CrawlerRunConfig(
			deep_crawl_strategy=BFSDeepCrawlStrategy(
				max_depth=1,
				include_external=False
			),
			verbose=True
		)

		async with AsyncWebCrawler() as crawler:
			results = await crawler.arun(main_url, config=config)

		if not results:
			print(f"⚠️ No matching subpages found for {main_url} - {category}")
			content = ""
			status = "no matching page"
		else:
			result = results[0]  # take the first matched result
			html = result.html
			main_text = extract_main_html_content(html)
			content = main_text[:config.max_content_length] if main_text else ""
			status = "success"
			print(f"✅ Extracted content from {result.url} for {main_url}")

	except Exception as e:
		print(f"❌ Error crawling {main_url}: {e}")
		content = ""
		status = "error"

	# Prepare update to sheet
	target_column = COLUMN_TO_WRITE_URL_TO.get(category.upper(), "K")
	content_range = f"{target_column}{row_index}"
	status_range = f"U{row_index}"
	updates = [
		{"range": content_range, "values": [[content]]},
		{"range": status_range, "values": [[status]]}
	]
	return updates

In [ ]:
# Main execution: Process all main URLs and batch update
print(f"\nProcessing {len(main_urls_with_rows)} main URLs in 1 batch")
all_updates = []
total_urls = len(main_urls_with_rows)
for idx, (row_index, main_url) in enumerate(main_urls_with_rows, 1):
	progress = idx / total_urls
	bar_length = 50
	filled = int(bar_length * progress)
	bar = '█' * filled + '-' * (bar_length - filled)
	percent = progress * 100
	print(f"Scraping websites: {percent:3.0f}% |{bar}| {idx}/{total_urls}")
	
	updates = await process_main_url(row_index, main_url, COLUMN_TO_PROCESS, FIRECRAWL_API)
	all_updates.extend(updates)
	print(f"Delaying for {config.delay_between_requests}s before next URL...")
	time.sleep(config.delay_between_requests)

# Perform batch update to Google Sheets with retries
print("\nPreparing to batch update Google Sheets...")
max_retries = config.max_retries
for attempt in range(max_retries):
	print(f"Attempt {attempt + 1}/{max_retries} to update Google Sheets")
	try:
		reader.service.spreadsheets().values().batchUpdate(
			spreadsheetId=spreadsheet_id,
			body={"valueInputOption": "RAW", "data": all_updates}
		).execute()
		print(f"Batch update successful for {len(all_updates)//2} rows")
		break
	except HttpError as e:
		print(f"HTTP Error on attempt {attempt + 1}: {e}")
		if attempt < max_retries - 1:
			wait_time = 5 * (attempt + 1)
			print(f"Retrying in {wait_time} seconds...")
			time.sleep(wait_time)
		else:
			print("All retries failed.")
	except Exception as e:
		print(f"Error during batch update on attempt {attempt + 1}: {e}")
		if attempt < max_retries - 1:
			print(f"Retrying in {5 * (attempt + 1)} seconds...")
			time.sleep(5 * (attempt + 1))
		else:
			print("All retries failed.")
print("Scraper execution complete")

[FETCH]... ↓ https://adaptfirst.com                                                                               |
✓ | ⏱: 6.70s 

[SCRAPE].. ◆ https://adaptfirst.com                                                                               |
✓ | ⏱: 0.03s 

[COMPLETE] ● https://adaptfirst.com                                                                               |
✓ | ⏱: 6.74s 

In [ ]:
# async def process_main_url(row_index: int, main_url: str, category: str, api_key: str):
# 	"""Process a single main URL: crawl, filter suburls, scrape, and prepare updates."""
# 	try:
# 		# Map the main URL to get list of subURLS
# 		app = FirecrawlApp(api_key=api_key)
# 		crawl_result = app.map_url(
# 			main_url
# 		)
# 		urls = crawl_result.links

# 		# Filter suburls based on category keywords
# 		categories = {
# 			"ABOUT_US": ["about", "who-we-are", "company", "our-story", "mission", "vision"],
# 			"EBOOK": ["ebook", "e-book", "downloads", "whitepaper", "guide", "brochure"],
# 			"COURSES": ["course", "training", "academy", "learning", "bootcamp"],
# 			"RECENT_BLOG": ["blog", "insights", "articles", "news", "stories"],
# 			"TESTIMONIALS": ["testimonial", "reviews", "feedback", "case-studies", "customers"],
# 			"WEBINAR": ["webinar", "events", "live", "sessions", "recording"],
# 			"SERVICES": ["service", "solutions", "offerings", "capabilities"],
# 			"PODCAST": ["podcast", "episodes", "listen", "audio"],
# 			"SHOP": ["shop", "store", "buy", "product", "checkout", "cart"]
# 		}
# 		keywords = categories.get(category.upper(), [])
# 		if not keywords:
# 			print(f"No keywords defined for column: {category}")
# 			return []
# 		filtered_urls = [
# 			url for url in urls
# 			if any(keyword in url.lower() for keyword in keywords)
# 		]

# 		# Scrape filtered suburls if any exist
# 		if filtered_urls:
# 			extractor = Crawl4AIExtractor(filtered_urls, max_concurrent=3)
# 			results = await extractor.extract_first_url()
# 			if results and len(results) > 0 and results[0]["status"] == "success":
# 				# Parse the HTML content from the first successful result
# 				html = results[0].get("content")
# 				main_text = extract_main_html_content(html)
# 				content = main_text[:config.max_content_length] if main_text else ""
# 				status = "success"
# 				print(f"{main_url} (found: {category.lower()}) - {status} - content extracted and parsed")
# 			else:
# 					content = ""
# 					status = "fail"
# 					print(f"{main_url} (found: {category.lower()}) - {status} - extraction failed")
# 		else:
# 				content = ""
# 				status = "no suburls"
# 				print(f"⚠️ {main_url} (found: {category.lower()}) - {status} - no matching suburls found")
# 	except Exception as e:
# 		error_msg = str(e).lower()
# 		if "insufficient credits" in error_msg or "payment required" in error_msg:
# 			print(f"⚠️ Firecrawl payment error for {main_url}: {e}")
# 			print("Please check your Firecrawl account at https://firecrawl.dev/pricing")
# 		else:
# 			print(f"Error processing {main_url}: {e}")
# 		content = ""
# 		status = "error"

# 	# Prepare updates for Google Sheets
# 	target_column = COLUMN_TO_WRITE_URL_TO.get(category.upper(), "K")
# 	content_range = f"{target_column}{row_index}"
# 	status_range = f"U{row_index}"
# 	updates = [
# 		{"range": content_range, "values": [[content]]},
# 		{"range": status_range, "values": [[status]]}
# 	]
# 	print(f"Prepared updates for row {row_index}: content to {content_range}, status to {status_range}")
# 	return updates

# # Main execution: Process all main URLs and batch update
# print(f"\nProcessing {len(main_urls_with_rows)} main URLs in 1 batch")
# # print("First few items in main_urls_with_rows:")
# # for i, item in enumerate(main_urls_with_rows[:3]):  # Show first 3 items
# #     print(f"Item {i}: {item} (type: {type(item)}, length: {len(item) if hasattr(item, '__len__') else 'N/A'})")
# all_updates = []
# total_urls = len(main_urls_with_rows)
# for idx, (row_index, main_url) in enumerate(main_urls_with_rows, 1):
# 	progress = idx / total_urls
# 	bar_length = 50
# 	filled = int(bar_length * progress)
# 	bar = '█' * filled + '-' * (bar_length - filled)
# 	percent = progress * 100
# 	print(f"Scraping websites: {percent:3.0f}% |{bar}| {idx}/{total_urls}")
	
# 	updates = await process_main_url(row_index, main_url, COLUMN_TO_PROCESS, FIRECRAWL_API)
# 	all_updates.extend(updates)
# 	print(f"Delaying for {config.delay_between_requests}s before next URL...")
# 	time.sleep(config.delay_between_requests)

# # Perform batch update to Google Sheets with retries
# print("\nPreparing to batch update Google Sheets...")
# max_retries = config.max_retries
# for attempt in range(max_retries):
# 	print(f"Attempt {attempt + 1}/{max_retries} to update Google Sheets")
# 	try:
# 		reader.service.spreadsheets().values().batchUpdate(
# 			spreadsheetId=spreadsheet_id,
# 			body={"valueInputOption": "RAW", "data": all_updates}
# 		).execute()
# 		print(f"Batch update successful for {len(all_updates)//2} rows")
# 		break
# 	except HttpError as e:
# 		print(f"HTTP Error on attempt {attempt + 1}: {e}")
# 		if attempt < max_retries - 1:
# 			wait_time = 5 * (attempt + 1)
# 			print(f"Retrying in {wait_time} seconds...")
# 			time.sleep(wait_time)
# 		else:
# 			print("All retries failed.")
# 	except Exception as e:
# 		print(f"Error during batch update on attempt {attempt + 1}: {e}")
# 		if attempt < max_retries - 1:
# 			print(f"Retrying in {5 * (attempt + 1)} seconds...")
# 			time.sleep(5 * (attempt + 1))
# 		else:
# 			print("All retries failed.")
# print("Scraper execution complete")